# SFT Instruct: De Modelo Base a Asistente General

Paso 1 del plan de SFT. Entrenamos el modelo base en Google Colab con el dataset Alpaca Spanish.

In [ ]:
# 1. Setup y dependencias
!pip install tiktoken datasets -q

from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/llm-desde-cero"
import os
os.makedirs(f"{PROJECT_DIR}/checkpoints", exist_ok=True)
print(f"Proyecto en: {PROJECT_DIR}")

In [ ]:
# 2. Arquitectura (GPT y Tokenizador)
import math
import time
import json
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
import tiktoken
import numpy as np

class TiktokenWrapper:
    def __init__(self, encoding_name: str = "cl100k_base"):
        self.encoding_name = encoding_name
        self.enc = tiktoken.get_encoding(encoding_name)
        self.tiktoken_to_compact = {}
        self.compact_to_tiktoken = {}
        self.vocab_size = 0

    def encode(self, text: str) -> list[int]:
        tids = self.enc.encode(text, allowed_special={"<|endoftext|>"})
        return [self.tiktoken_to_compact[t] for t in tids if t in self.tiktoken_to_compact]

    def decode(self, compact_ids: list[int]) -> str:
        tids = [self.compact_to_tiktoken.get(c, 0) for c in compact_ids]
        return self.enc.decode(tids)

    def load(self, path: str):
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        self.encoding_name = data["encoding_name"]
        self.enc = tiktoken.get_encoding(self.encoding_name)
        self.vocab_size = data["vocab_size"]
        self.tiktoken_to_compact = {int(k): v for k, v in data["tiktoken_to_compact"].items()}
        self.compact_to_tiktoken = {int(k): v for k, v in data["compact_to_tiktoken"].items()}
        print(f"[Tokenizer] Vocab cargado ({self.vocab_size:,} tokens)")

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.n_head, self.n_embd = n_head, n_embd
        self.d_k = n_embd // n_head
        self.dropout_p = dropout
        self.qkv_proj = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.out_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv_proj(x)
        Q, K, V = qkv.split(self.n_embd, dim=2)
        Q = Q.view(B, T, self.n_head, self.d_k).transpose(1, 2)
        K = K.view(B, T, self.n_head, self.d_k).transpose(1, 2)
        V = V.view(B, T, self.n_head, self.d_k).transpose(1, 2)
        out = F.scaled_dot_product_attention(Q, K, V, dropout_p=self.dropout_p if self.training else 0.0, is_causal=True)
        out = out.transpose(1, 2).contiguous().view(B, T, self.n_embd)
        return self.resid_dropout(self.out_proj(out))

class FeedForward(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), nn.GELU(),
            nn.Linear(4 * n_embd, n_embd), nn.Dropout(dropout))
    def forward(self, x): return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = MultiHeadSelfAttention(n_embd, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffn = FeedForward(n_embd, dropout)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self, vocab_size, n_embd, n_head, n_layer, block_size, dropout):
        super().__init__()
        self.block_size = block_size
        self.transformer = nn.ModuleDict({
            'tok_emb': nn.Embedding(vocab_size, n_embd),
            'pos_emb': nn.Embedding(block_size, n_embd),
            'drop':    nn.Dropout(dropout),
            'blocks':  nn.ModuleList([TransformerBlock(n_embd, n_head, block_size, dropout) for _ in range(n_layer)]),
            'ln_f':    nn.LayerNorm(n_embd),
        })
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.transformer['tok_emb'].weight = self.lm_head.weight

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(0, T, device=idx.device)
        x = self.transformer['drop'](self.transformer['tok_emb'](idx) + self.transformer['pos_emb'](pos))
        for block in self.transformer['blocks']:
            x = checkpoint(block, x, use_reentrant=False)
        x = self.transformer['ln_f'](x)
        if targets is not None:
            chunk_size = 128
            loss = torch.tensor(0.0, device=idx.device)

            for i in range(0, T, chunk_size):
                x_chunk = x[:, i:i+chunk_size, :]
                t_chunk = targets[:, i:i+chunk_size]
                logits_chunk = self.lm_head(x_chunk)
                loss += F.cross_entropy(
                    logits_chunk.view(-1, logits_chunk.size(-1)),
                    t_chunk.contiguous().view(-1),
                    ignore_index=-100,
                    reduction="sum"
                )
            valid_tokens = (targets != -100).sum().clamp(min=1)
            loss = loss / valid_tokens
            logits = None
        else:
            logits = self.lm_head(x)
            loss = None

        return logits, loss


In [ ]:
# 3. Cargar Tokenizador y Dataset Alpaca
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader

tokenizer = TiktokenWrapper()
tokenizer.load(f"{PROJECT_DIR}/data/vocab.json")

print("[Data] Descargando Alpaca Spanish...")
dataset = load_dataset("bertin-project/alpaca-spanish", split="train")


def format_example(example):
    instruction = example['instruction']
    input_text = example.get('input', '')
    output_text = example['output']

    if input_text:
        prompt = f"### Instrucción:\n{instruction}\n{input_text}\n### Respuesta:\n"
    else:
        prompt = f"### Instrucción:\n{instruction}\n### Respuesta:\n"

    response = f"{output_text}\n###"
    return prompt, response

class SFTDataset(Dataset):
    def __init__(self, data, tokenizer, max_seq_len):
        self.data = data
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        prompt, response = format_example(self.data[idx])
        prompt_ids = self.tokenizer.encode(prompt)
        response_ids = self.tokenizer.encode(response)

        # Construir secuencia completa
        full_ids = prompt_ids + response_ids
        if len(full_ids) > self.max_seq_len + 1:
            full_ids = full_ids[:self.max_seq_len + 1]

        # Construir targets: -100 para el prompt, IDs reales para la response
        targets = [-100] * len(prompt_ids) + response_ids
        if len(targets) > self.max_seq_len + 1:
            targets = targets[:self.max_seq_len + 1]

        # Padding si es más corto
        pad_len = (self.max_seq_len + 1) - len(full_ids)
        if pad_len > 0:
            full_ids = full_ids + [0] * pad_len
            targets = targets + [-100] * pad_len

        x = torch.tensor(full_ids[:-1], dtype=torch.long)
        y = torch.tensor(targets[1:], dtype=torch.long)

        # Evitar target con todo -100 que causa loss = NaN
        if (y != -100).sum() == 0:
            y[-1] = 0

        return x, y

# Dividir en train/val
dataset = dataset.train_test_split(test_size=0.05, seed=42)
train_data = dataset['train']
val_data = dataset['test']

max_seq_len = 512
batch_size = 8
train_ds = SFTDataset(train_data, tokenizer, max_seq_len)
val_ds = SFTDataset(val_data, tokenizer, max_seq_len)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, pin_memory=True)
print(f"[Data] Train: {len(train_ds)} | Val: {len(val_ds)}")

In [ ]:
# 4. Configuración SFT Instruct
config = {
    "n_embd"      : 1024,
    "n_head"      : 16,
    "n_layer"     : 24,
    "block_size"  : 1024,
    "dropout"     : 0.1,

    "learning_rate": 1e-5,        # 30x menor que pretraining
    "num_epochs"  : 3,
    "grad_accum"  : 2,
    "warmup_steps": 100,
    "weight_decay": 0.01,
    "grad_clip"   : 1.0,
    "eval_interval": 500,
}

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[SFT] Device: {device}")

model = GPT(
    vocab_size=tokenizer.vocab_size, n_embd=config["n_embd"],
    n_head=config["n_head"], n_layer=config["n_layer"],
    block_size=config["block_size"], dropout=config["dropout"]
).to(device)

ckpt_path = f"{PROJECT_DIR}/checkpoints/best_model.pt"
print(f"[SFT] Cargando weights preentrenados de {ckpt_path}...")
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt["model_state"])

print("[SFT] Compilando modelo...")
model = torch.compile(model)


In [ ]:
# 5. Bucle de Entrenamiento SFT
from torch.amp import autocast

optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"])

def get_lr(step, total_steps):
    if step < config["warmup_steps"]:
        return config["learning_rate"] * step / config["warmup_steps"]
    decay_ratio = (step - config["warmup_steps"]) / (total_steps - config["warmup_steps"])
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return config["learning_rate"] * coeff

@torch.no_grad()
def evaluate():
    model.eval()
    losses = []
    for x, y in val_loader:
        x, y = x.to(device), y.to(device)
        with autocast(device_type="cuda", dtype=torch.bfloat16):
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return sum(losses) / len(losses)

total_steps = len(train_loader) * config["num_epochs"] // config["grad_accum"]
print(f"[SFT] Total steps: {total_steps} (Epochs: {config['num_epochs']})")

step = 0
best_val_loss = float('inf')
model.train()

for epoch in range(config["num_epochs"]):
    optimizer.zero_grad()
    for i, (x, y) in enumerate(train_loader):
        lr = get_lr(step, total_steps)
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr

        x, y = x.to(device), y.to(device)
        with autocast(device_type="cuda", dtype=torch.bfloat16):
            _, loss = model(x, y)
            loss = loss / config["grad_accum"]

        loss.backward()

        if (i + 1) % config["grad_accum"] == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip"])
            optimizer.step()
            optimizer.zero_grad()
            step += 1

            if step % 50 == 0:
                print(f"Epoch {epoch+1}/{config['num_epochs']} | Step {step}/{total_steps} | Loss: {loss.item() * config['grad_accum']:.4f} | LR: {lr:.2e}")

            if step % config["eval_interval"] == 0:
                val_loss = evaluate()
                print(f"--> Val Loss: {val_loss:.4f}")
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    inner = getattr(model, '_orig_mod', model)
                    torch.save({"model_state": inner.state_dict()}, f"{PROJECT_DIR}/checkpoints/instruct_model.pt")
                    print("    (Modelo guardado!)")

# Guardar modelo final al terminar
inner = getattr(model, '_orig_mod', model)
torch.save({"model_state": inner.state_dict()}, f"{PROJECT_DIR}/checkpoints/instruct_model.pt")
print("[SFT] Completado y guardado en checkpoints/instruct_model.pt")